# ANOVA(분산분석)
- 자, 여러분. 우리가 비교해야 하는 집단이 두개면 티테스트나 맨휫흐니나 카이스퀘어를 쓰면 되잖아요? 근데 분산분석은 세 개 이상일 때 쓰는 방법입니다. 
- 다만 분산분석의 경우 '얘네들 중 뭔가 다른 게 있다'만 나오는거기떄문에 후속 분석을 진행해야 합니다. 
- 일원분산분석(요인이 하나)의 경우 대응되는 비모수 방법으로 크러스칼-월리스 검정이라는 게 있습니다. 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats # 얘 써도 되고요 
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.graphics.factorplots import interaction_plot

import statsmodels.api as sm # 얘도 됩니다. 코드가 좀 다름. 
from statsmodels.formula.api import ols

# 일원분산분석(1-way anova)

In [ ]:
# 예시 데이터
fert_a = np.random.normal(10, 2, 30)
fert_b = np.random.normal(10, 2, 30)
fert_c = np.random.normal(13, 2, 30)

## 가설
- 귀무가설: 모든 그룹이 다 똑같다(=그놈이 그놈이다, 또이또이)
- 대립가설: 얘네들 중 적어도 하나는 다른 게 있다. 

In [ ]:
# ANOVA
f_stat, p_val = stats.f_oneway(fert_a, fert_b, fert_c)

print(f'p-value: {p_val:.4e}')

if p_val > 0.05:
    print('귀무가설이 기각되지 않았습니다. ')
else: 
    print('튜키 드가자!!!')

In [ ]:
# 데이터프레임 생성
df_anova = pd.DataFrame({
    # 성장 데이터 합치기
    'Growth' : np.concatenate([fert_a, fert_b, fert_c]),
    # 그룹
    'Fertilizer' : ['A'] * 30 + ['B'] * 30 + ['C'] * 30
})

## Tukey HSD
- ANOVA는 얘네들 중 다른 게 있다! 만 알려주기떄문에 구체적으로 뭐가 다른지를 찾으려면 후속 분석이 필요하다고 했죠? 그 후속 분석 중 하나가 Tukey HSD입니다. 저는 튜키라고 합니다. 
- 후속 분석은 ANOVA에서 귀무가설을 기각했을때만 하시면 됩니다. 위 분석에서 P-value가 유의수준보다 크다면 얘를 할 필요가 없어요. 
~~튜키씨 미안... 당신 스펠링 헷갈려...~~

In [ ]:
tukey_fert = pairwise_tukeyhsd(endog=df_anova['Growth'],      # 결과값 (수치형)
                            groups=df_anova['Fertilizer'],  # 비교 집단 (범주형)
                            alpha=0.05)

print(tukey_fert)

- 저기서 True가 뜨는 게 얘랑 얘랑 다르더라~ 이런 겁니다. 

In [ ]:
fig = tukey_fert.plot_simultaneous(figsize=(10, 6))
plt.title("Tukey HSD: Fertilizer comparison")
plt.axvline(np.mean(fert_c), linestyle='--')
plt.xlabel("Growth")
plt.show()

# 이원분산분석(2-way anova)
- 독립변수가 두개지요. 

In [ ]:
# 데이터 생성
data = pd.DataFrame({
    'Drug': ['A']*20 + ['B']*20,
    'Exercise': (['High']*10 + ['Low']*10) * 2,
    'Weight_Loss': [
        # Drug A: High(7~9), Low(3~5)
        *np.random.normal(8, 1, 10), *np.random.normal(4, 1, 10),
        # Drug B: High(6~8), Low(5~7)
        *np.random.normal(7, 1, 10), *np.random.normal(6, 1, 10)
    ]
})

- 여기서는 독립변수가 성별, 그리고 약물 두 개입니다. 이럴때 일원분석을 하면 결과 망해요. 그걸 어떻게 아냐고요? 해봤음... 

In [ ]:
model = ols('Weight_Loss ~ C(Drug) * C(Exercise)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print(anova_table)

- 저기서 PR이 P-value인데, 보시면 다 0.05보다는 작은 것 같죠? 그럼 튜키를 해야죠. 

## Interaction plot

In [ ]:
# 폰트는 이미 NanumSquare로 설정되어 있으니 바로 그립니다.
fig = interaction_plot(x=data['Exercise'], 
                        trace=data['Drug'], 
                        response=data['Weight_Loss'],
                        colors=['red', 'blue'], 
                        markers=['D', '^'])

plt.title('Weight Loss Interaction: Drug & Exercise')
plt.show()

- 이건 또 머고... 아니 설명하는 양반이 그것도 몰라요? 네. 저도 오늘 처음봅니다. 전에 했던 이원분산분석은 양상을 시각화를 통해서 확인하고 통계적으로 유의한지 확인하기 위한 거라 이걸 안 그렸어요. 
- 가로축은 운동 강도, 세로축은 체중 감량 평균입니다. 선의 기울기는 운동 강도에 따라 평균이 어떻게 변화하는지라고 보시면 되고... 
- 저기 선이 교차하고 있죠? 네, 보조제와 운동 두 효과가 상호작용을 하고 있다는 애기입니다. 밑에 있는 튜키 결과를 보시면 빡세게 운동하는 그룹은 별로 차이 없었지만(튜키에서 폴스뜸) 운동 저강도로 하는 그룹은 보조제에 따라 차이가 있었어요. ~~아래 블록에 튜키 있어요~~
- 만약 두 요인이 서로 영향을 주지 않을 경우, 평행선을 그리게 됩니다. 

## Tukey HSD

In [ ]:
# 1. '약 + 운동' 조합 컬럼 만들기 (사후검정을 위해 그룹을 하나로 합침)
data['Combination'] = data['Drug'] + " / " + data['Exercise']

# 2. Tukey HSD 실행
tukey = pairwise_tukeyhsd(endog=data['Weight_Loss'],     # 종속변수
                            groups=data['Combination'],   # 그룹화 변수
                            alpha=0.05)                   # 유의수준

print("=== Tukey HSD 사후검정 결과 ===")
print(tukey)

- 상호작용 여부까지 확인한 튜키

In [ ]:
# 운동
print("=== 운동량별 차이 (Tukey HSD) ===")
tukey_exec = pairwise_tukeyhsd(endog=data['Weight_Loss'], groups=data['Exercise'], alpha=0.05)
print(tukey_exec)

# 보조제
print("\n=== 보조제별 차이 (Tukey HSD) ===")
tukey_drug = pairwise_tukeyhsd(endog=data['Weight_Loss'], groups=data['Drug'], alpha=0.05)
print(tukey_drug)

- 얘는 다른 요인을 배제하고 이 요인에 따라 다른가? 만 본 겁니다. 만약 이원분산분석 했는데 p-value가 다 유의수준 이하면 둘 다 보셔야 합니다. 
- 근데 둘다 피밸류 낮게 나와서 봤더니 튜키에서 또이또이로 판명되는 경우도 있음... 

In [ ]:
# Tukey 결과 시각화
fig = tukey.plot_simultaneous()
plt.title('Tukey HSD Multiple Comparison')
plt.xlabel('Weight Loss (kg)')
plt.show()